# Membuat sparksession

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, rank, row_number, when
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Tugas5") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


# Menyiapkan data

**Dataset Transaksi**

In [3]:
df_transaksi = spark.read.csv("hdfs://localhost:9000//user/asfadani/tugas5/transaksi_tugas5.csv", header=True, inferSchema=True)

# Menampilkan 10 data awal, skema, dan jumlah baris
df_transaksi.show(10)

+--------+--------------------+----------+------------+------------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|
+--------+--------------------+----------+------------+------------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|
|   TRX-5|Kesehatan & Kecan...|  Magelang|           4|      100000|
|   TRX-6|        Rumah Tangga|  Magelang|           6|       25000|
|   TRX-7|          Elektronik|  Semarang|           8|       50000|
|   TRX-8|             Fashion|  Semarang|           5|       25000|
|   TRX-9|          Elektronik|Yogyakarta|           3|      100000|
+--------+--------------------+----------+------------+------------+
only showing top 10 rows



In [4]:
# Menambahkan kolom pendapatan
df_transaksi = df_transaksi.withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))
df_transaksi.show(10)

+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    150000|
|   TRX-5|Kesehatan & Kecan...|  Magelang|           4|      100000|    400000|
|   TRX-6|        Rumah Tangga|  Magelang|           6|       25000|    150000|
|   TRX-7|          Elektronik|  Semarang|           8|       50000|    400000|
|   TRX-8|             Fashion|  Semarang|           5|       25000|    125000|
|   TRX-9|          Elektronik|Yogyakart

**Dataset target cabang**

In [5]:
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

df = pd.DataFrame(data_target_cabang)

df_targer_cabang = spark.createDataFrame(df)
df_targer_cabang.show()

+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      Joko|
|  Semarang|      55000000|      Sari|
|      Solo|      40000000|      Bayu|
| Purworejo|      30000000|     Fitri|
+----------+--------------+----------+



# Join dan perbandingan target

membuat ringkasan total pendapatan per kota, lalu join dengan dataset target cabang nya. Dan menambahkan kolom pencapaian dalam persen nya.

**Membuat ringkasan per kota**

In [6]:
df_transaksi_ringkas = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan"),
    avg("pendapatan").alias("rata_rata_pendapatan"),
    count("*").alias("jumlah_transaksi")
)

df_transaksi_ringkas.show()

+----------+----------------+--------------------+----------------+
|      kota|total_pendapatan|rata_rata_pendapatan|jumlah_transaksi|
+----------+----------------+--------------------+----------------+
|  Magelang|        31650000|  368023.25581395347|              86|
|  Semarang|        38175000|  410483.87096774194|              93|
|      Solo|        33475000|   352368.4210526316|              95|
| Purworejo|        45650000|   393534.4827586207|             116|
|Yogyakarta|        47275000|   429772.7272727273|             110|
+----------+----------------+--------------------+----------------+



**Join dengan dataset target cabang**

In [7]:
df_gabung = df_targer_cabang.join(df_transaksi_ringkas, on="kota", how="left")

df_gabung.show()

+----------+--------------+----------+----------------+--------------------+----------------+
|      kota|target_bulanan|pic_cabang|total_pendapatan|rata_rata_pendapatan|jumlah_transaksi|
+----------+--------------+----------+----------------+--------------------+----------------+
|  Magelang|      45000000|      Rani|        31650000|  368023.25581395347|              86|
|Yogyakarta|      60000000|      Joko|        47275000|   429772.7272727273|             110|
|  Semarang|      55000000|      Sari|        38175000|  410483.87096774194|              93|
|      Solo|      40000000|      Bayu|        33475000|   352368.4210526316|              95|
| Purworejo|      30000000|     Fitri|        45650000|   393534.4827586207|             116|
+----------+--------------+----------+----------------+--------------------+----------------+



**Menambah kolom pencapaian dalam %**

In [8]:
df_gabung = df_gabung.withColumn("pencapaian_persen", (col("total_pendapatan") / col("target_bulanan")) * 100)

df_gabung.show()

+----------+--------------+----------+----------------+--------------------+----------------+------------------+
|      kota|target_bulanan|pic_cabang|total_pendapatan|rata_rata_pendapatan|jumlah_transaksi| pencapaian_persen|
+----------+--------------+----------+----------------+--------------------+----------------+------------------+
|  Magelang|      45000000|      Rani|        31650000|  368023.25581395347|              86| 70.33333333333334|
|Yogyakarta|      60000000|      Joko|        47275000|   429772.7272727273|             110| 78.79166666666667|
|  Semarang|      55000000|      Sari|        38175000|  410483.87096774194|              93|  69.4090909090909|
|      Solo|      40000000|      Bayu|        33475000|   352368.4210526316|              95|           83.6875|
| Purworejo|      30000000|     Fitri|        45650000|   393534.4827586207|             116|152.16666666666669|
+----------+--------------+----------+----------------+--------------------+----------------+---

# Window Function - Kategori terlaris tiap kota

Menerapkan window function untuk menentukan ranking kategori pendapatan tertinggi di tiap kota. Menggunakan `row_number()`.

In [9]:
from pyspark.sql.functions import sum as spark_sum, col

pendapatan_kat = df_transaksi.groupBy("kota", "kategori").agg(
    spark_sum(col("pendapatan")).alias("pendapatan_kategori")
)

In [10]:
window_spec = Window.partitionBy("kota").orderBy(col("pendapatan_kategori").desc())

df_ranking = pendapatan_kat.withColumn(("rank_kategori_laris"), row_number().over(window_spec))

df_ranking.filter(col("rank_kategori_laris") == 1) \
    .drop("rank_kategori_laris") \
    .show()

+----------+--------------------+-------------------+
|      kota|            kategori|pendapatan_kategori|
+----------+--------------------+-------------------+
|  Magelang|Kesehatan & Kecan...|            7275000|
| Purworejo|Kesehatan & Kecan...|           10075000|
|  Semarang|        Rumah Tangga|           11125000|
|      Solo|Kesehatan & Kecan...|            8425000|
|Yogyakarta|             Fashion|           13325000|
+----------+--------------------+-------------------+



# SparkSQL

Menjadikan dataframe transaksi dan target menjadi tabel sementara dan mengeksekusi dengan kueri SQL. Menampilkan kota, pic_cabang, dan jumlah transaksi (`COUNT`) di kota tersebut dan diurutkan dari jumlah transaksi terbanyak.

In [9]:
df_transaksi.createOrReplaceTempView("transaksi")
df_targer_cabang.createOrReplaceTempView("target")

print("mengubah dataframe ke tabel sementara")

mengubah dataframe ke tabel sementara


In [10]:
kueri_tugas = spark.sql("""
    SELECT t.kota, ta.pic_cabang, COUNT(t.unit_terjual) AS jumlah_transaksi
    FROM transaksi t
    JOIN target ta ON t.kota = ta.kota
    GROUP BY t.kota, ta.pic_cabang
    ORDER BY jumlah_transaksi DESC
""")
kueri_tugas.show()

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



# Kesimpulan

berdasarkan hasil bagian A dan B, **cabang mana yang berkinerja paling baik** dan **cabang mana yang paling perlu perhatian manajemen**? Sertakan angka-angka pendukung dari hasil analisis kalian, bukan opini tanpa dasar data.

>Berdasarkan data yang ditampilkan Kota Magelang menjadi kota yang perlu diperhatikan karena jumlah transaksi yang terjadi menjadi yang paling sedikit yang hanya 86 transaksi. Meskipun persentasi pencapaiannya sudah di angka yang cukup tinggi dengan 70% pencapaian dari target. Selain itu Semarang juga menjadi kota yang harus lebih diperhatikan lagi meskipun jumlah transaksinya mencapai 93 transaksi lenih banyak dari transaksi kota Magelang. Tetapi dikarenakan target bulanan yang harus dicapai sangat tinggi yaitu Rp 55 juta membuat hasil transaksi yang tercapai belum sesuai target. Dapat dilihat dari presentase capaian nya yang hanya 69% dari target bulanan yang telkah ditentukan.
Untuk kota dengan kinerja yang baik ada di kota Purworejo dengan jumlah transaksi yang paling tinggi yaitu 116 transaksi dan juga capaian nya mencapai 152% jauh melebihi target bulanan yang telah ditetapkan. Selain itu ada kota Yogyakarta dengan jumlah transaksi 110 transaksi yang menempati peringkat ke 2 kota dengan transaksi tertinggi serta presentase capaiannya yang sudah tinggi juga di angka 78%. Selain itu kota solo juga menjadi salah satu kota dengan kinerja yang baik dengan capaiannya mencapai 83% dari target yang telah ditentukan.



**Menutu spark session**

In [11]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
